<a href="https://colab.research.google.com/github/rendzina/BigDataAndVisualisation/blob/main/Colab/Hello_World_CoLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hello World
## Written for CoLab

This notebook demonstrates reading HVAC sensor data from Azure Blob storage, exploring it with Spark DataFrames, and querying it with SQL.

### Mount Azure Blob storage

The sample data lives in Azure Blob storage. We install the mount package and then mount the storage so the notebook can read the HVAC CSV file.

**Install the mount-azure-blob package** (run once per session).

**Mount the storage** using the connection details given in class. You will be prompted for account name and key.

In [28]:
%%capture
%pip install mount-azure-blob==0.0.3

In [29]:
# Connection details are given in class
from mount_azure_blob import mount_storage
mount_storage(mount_path="bdv-2024-05-09t15-59-02-855z", config_file=None)


Mounting azure blob storage...
Dependencies Installed...
Successfully Mounted...


### Initialise Spark

Create a Spark session so we can use the DataFrame API and SQL. This is the entry point for Spark in the notebook.

In [ ]:
import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("BDV Spark Example") \
       .getOrCreate()

spark

**Import** CSV and string I/O utilities (used when working with raw text or CSV content).

In [31]:
import csv
import io
from io import StringIO

**Read the HVAC CSV** from the mounted blob path into a Spark DataFrame. We use the first row as the header and let Spark infer column types.

In [32]:
# Create a Spark dataframe and table from sample data (note this is NOT a Pandas dataframe)
csvFile = spark.read.csv('bdv-2024-05-09t15-59-02-855z/HdiSamples/HdiSamples/SensorSampleData/hvac/HVAC.csv', header=True, inferSchema=True)

**Row count:** check how many records are in the DataFrame.

In [33]:
# How many rows in dataframe?
csvFile.count()

8000

### HVAC dataset description

The **HVAC** (Heating, Ventilation, and Air Conditioning) sample data contains sensor readings from buildings: date and time, target and actual temperatures, system identifier, system age, and building ID. We use it to explore DataFrames and later to run SQL queries.

In [ ]:
# Show the schema: column names and types inferred by Spark
csvFile.printSchema()

In [ ]:
# Preview the first 10 rows
csvFile.show(10)

In [ ]:
# Summary statistics for numeric columns
csvFile.describe().show()

### Register the DataFrame as a SQL temp view

We register `csvFile` as a temporary view so we can query it with Spark SQL in the following cells.

In [34]:
csvFile.createOrReplaceTempView("hvac_ss01shh")

**Verify row count** using SQL (should match the DataFrame count above).

In [35]:
spark.sql("SELECT count(*) from hvac_ss01shh").show()

+--------+
|count(1)|
+--------+
|    8000|
+--------+



**Describe the table** to list columns and their types in SQL catalog form.

In [36]:
spark.sql("DESCRIBE TABLE hvac_ss01shh").show()


+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|      Date|   string|   null|
|      Time|   string|   null|
|TargetTemp|      int|   null|
|ActualTemp|      int|   null|
|    System|      int|   null|
| SystemAge|      int|   null|
|BuildingID|      int|   null|
+----------+---------+-------+



**Example query:** temperature difference (target − actual) for a given date. Positive values mean the system was heating; negative means cooling.

In [46]:
spark.sql("SELECT 'buildingID', (targettemp - actualtemp) AS temp_diff, date FROM hvac_ss01shh WHERE date = '6/1/13'").show()

+----------+---------+------+
|buildingID|temp_diff|  date|
+----------+---------+------+
|buildingID|        8|6/1/13|
|buildingID|        2|6/1/13|
|buildingID|      -10|6/1/13|
|buildingID|        3|6/1/13|
|buildingID|        9|6/1/13|
|buildingID|        5|6/1/13|
|buildingID|       11|6/1/13|
|buildingID|       -7|6/1/13|
|buildingID|       14|6/1/13|
|buildingID|       -3|6/1/13|
|buildingID|       -8|6/1/13|
|buildingID|       -1|6/1/13|
|buildingID|       11|6/1/13|
|buildingID|       14|6/1/13|
|buildingID|       -4|6/1/13|
|buildingID|        4|6/1/13|
|buildingID|        4|6/1/13|
|buildingID|       12|6/1/13|
|buildingID|       -9|6/1/13|
|buildingID|      -10|6/1/13|
+----------+---------+------+
only showing top 20 rows

